# Final_project — 택배 분류 연구 실험실

Windows 노트북의 VS Code에서 **Google Colab GPU 커널**로 실행합니다. 로봇은 원격 Linux의 MuJoCo에서 움직이고, 영상은 노트북 안에 표시합니다. 로컬 PC에 GPU나 MuJoCo 창이 필요하지 않습니다.

**먼저 할 일:** VS Code에 Google의 Colab 확장과 Jupyter 확장을 설치 → 이 파일 열기 → 오른쪽 위 `Select Kernel → Colab`에서 사용 가능한 GPU 서버 선택 → Google 로그인. `Auto Connect`가 GPU를 보장하지 않으므로 아래 진단 결과를 확인합니다.

로컬 Windows 파일과 Colab `/content`는 서로 다른 컴퓨터의 파일입니다. 로컬에서 수정한 `.py`는 자동으로 원격에 반영되지 않습니다. Git으로 push한 브랜치를 설정하거나, Colab 확장의 `Upload to Colab`으로 올린 뒤 그 원격 폴더를 지정합니다. 노트북을 저장하는 것만으로 원격 데이터와 weight가 보존되지는 않습니다.

실행 순서: **설정 → 코드·저장소 → 설치 → 기본 동작 → 데이터 수집 → VLM 학습·재로딩 → VLA 학습·재로딩 → 연구 실험 → 결과 보관**. 처음에는 `connection` 프로필을 끝까지 실행합니다. 1~2스텝 학습은 연결 검사이며 완성된 분류 모델이 아닙니다.

## 학생이 책임질 판단

1. 어떤 실패를 연구할지: 파지 여유, 목적지 점유, 실패 후 재시도 중 핵심 하나를 고릅니다.
2. 왜 효과가 있을지: 비교군·예측·반대 결과·같은 정보와 예산을 먼저 적습니다.
3. 무엇을 바꿀지: 접근성 기준 / 다음 물체 선택 / 재계획 조건을 스스로 설계합니다.
4. 어떻게 확인할지: 독립 배치, 실패를 포함한 완료율과 비용, 반례를 정합니다.
5. 결과가 다르면 무엇을 할지: 경쟁 설명과 이를 구분할 다음 실험을 제안합니다.

조교는 설치·저장·제어·학습 입출력·측정 기능을 제공합니다. 아래 기본 경로는 학생 설계 빈칸을 채우기 전에도 실행됩니다. 개념은 `docs/STUDENT_RESEARCH_GUIDE.md`, 설계 기록은 `docs/WORKSHEETS.md`를 함께 사용하세요.

현재 과제는 **평면 단순 물체·고정 목표 자리**입니다. 진짜 적층의 지지 관계, 자유 임시 보관, 동적 유입은 추가 구현과 검증이 필요한 확장입니다.


## 1. 실행 설정 — 실험 전에 고정하기

`connection`은 적은 데이터·짧은 학습의 연결 확인입니다. `study`로 바꾸는 것만으로 과학적 실험이 완성되지는 않습니다. 아래 학습량·장면 수는 시작값이며 validation과 예비시험으로 정하고 test는 마지막에만 사용합니다.

모든 결과는 새 RUN_ID에 저장됩니다. 기본은 Google Drive의 `MyDrive/VLA2026/Final_project/팀이름/RUN_ID`입니다. 공유 Drive를 쓰려면 `PERSIST_BASE`를 실제 마운트된 공유 폴더로 바꾸세요. 동일 폴더를 덮어쓰지 않습니다.


In [ ]:
import os, sys, json, platform, shutil, subprocess, uuid, re, time
from pathlib import Path
from datetime import datetime, timezone
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
TEAM = "team01"                          # 영문/숫자/_/-로 팀 이름
PROFILE = "connection"                   # connection 또는 study
REPO_URL = "https://github.com/semin126/VLA2026.git"
REPO_REF = "main"                        # 학생 브랜치 또는 검토한 commit
SOURCE_ROOT = os.environ.get("FINAL_PROJECT_SOURCE", "")  # 원격 vla_pipeline 경로. 비우면 clone
USE_DRIVE = os.environ.get("FINAL_PROJECT_NO_DRIVE") != "1"
INSTALL = os.environ.get("FINAL_PROJECT_SKIP_INSTALL") != "1"
RUN_ML = os.environ.get("FINAL_PROJECT_NO_ML") != "1"
PERSIST_BASE = "/content/drive/MyDrive/VLA2026/Final_project"
WORK_BASE = Path(os.environ.get("FINAL_PROJECT_WORK", "/content/Final_project_runs"))
PYTHON_OVERRIDE = os.environ.get("FINAL_PROJECT_PYTHON", "")  # 조교 검사 전용
VLM_MODEL = "HuggingFaceTB/SmolVLM2-500M-Video-Instruct"
PIPELINE_CHECK_VLM = "Qwen/Qwen2.5-VL-3B-Instruct"  # 학습하지 않은 연결 확인용 별도 모델
VLA_BASE = "lerobot/smolvla_base"
VLM_STEPS = 1 if PROFILE == "connection" else 100
VLA_STEPS = 2 if PROFILE == "connection" else 2000
VLA_BATCH = 1                            # 작은 GPU부터. 비교 시 실제 batch도 기록
COUNT = 10
COLLECT_SEED = 3
COLLECT_SCENES = 3 if PROFILE == "connection" else 12
HOLDOUT_SEEDS = [5] if PROFILE == "connection" else [13, 14]
VLM_TRAIN_SEEDS = "840" if PROFILE == "connection" else "800:820"
VLM_VAL_SEEDS = "841" if PROFILE == "connection" else "840:845"
VLM_TEST_SEEDS = "842" if PROFILE == "connection" else "860:870"
EVALUATE_TEST = False                    # 탐색 중에는 validation. 최종 고정 후에만 True
assert PROFILE in {"connection", "study"}
assert re.fullmatch(r"[A-Za-z0-9_-]+", TEAM)
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
RUN = WORK_BASE / TEAM / RUN_ID
RUN.mkdir(parents=True, exist_ok=False)
print("RUN_ID:", RUN_ID, "\n임시 작업 폴더 / work:", RUN)
print("kernel:", platform.system(), sys.version.split()[0])


## 2. Drive 저장 위치 확인

인증 화면이 나오면 본인의 Google 계정으로 허용하세요. **비밀번호나 인증 코드를 노트북에 적지 않습니다.** VS Code 확장의 `Colab: Mount Google Drive to Server...`도 사용할 수 있습니다. 마운트가 실패하면 이 셀에서 멈추며, 저장되지 않는 상태로 학습을 계속하지 않습니다. 명시적으로 `USE_DRIVE=False`를 택한 로컬 검사는 영구 저장 검증이 아닙니다.


In [ ]:
if USE_DRIVE:
    if platform.system() != "Linux":
        raise RuntimeError("Colab 커널을 선택하세요. 현재 Windows 로컬 커널입니다.")
    if not Path("/content/drive/MyDrive").is_dir():
        from google.colab import drive
        drive.mount("/content/drive")
    if not Path("/content/drive/MyDrive").is_dir():
        raise RuntimeError("Drive가 보이지 않습니다. 마운트와 권한을 확인하세요.")
    PERSIST = Path(PERSIST_BASE) / TEAM / RUN_ID
else:
    PERSIST = RUN.parent / (RUN.name + "_local_backup")
    print("명시적 로컬 검사: Google Drive 영구 저장은 검증하지 않습니다.")
PERSIST.mkdir(parents=True, exist_ok=False)
if PERSIST.resolve().is_relative_to(RUN.resolve()):
    raise ValueError("보관 폴더는 임시 RUN 폴더 밖에 지정하세요.")
probe = PERSIST / "WRITE_TEST.txt"
probe.write_text(RUN_ID, encoding="utf-8")
assert probe.read_text(encoding="utf-8") == RUN_ID
print("영구 보관 위치 / backup:", PERSIST)


## 3. 원격 코드 준비와 기록

저장소에는 이 노트북과 모듈이 함께 있어야 합니다. 아직 최신 코드가 push되지 않았다면 준비된 원격 `vla_pipeline` 폴더를 `SOURCE_ROOT`에 지정합니다. 경로에는 Windows `C:\...`를 넣지 않습니다. 이미 받은 Git 폴더를 자동 pull/reset하지 않으며, 새 실행에는 새 clone을 사용합니다.


In [ ]:
if SOURCE_ROOT:
    PROJECT = Path(SOURCE_ROOT).expanduser().resolve()
else:
    checkout = RUN / "checkout"
    subprocess.run(["git", "clone", REPO_URL, str(checkout)], check=True)
    subprocess.run(["git", "-C", str(checkout), "checkout", REPO_REF], check=True)
    PROJECT = checkout / "vla_pipeline"
required = ["setup_assets.py", "requirements-colab-lock.txt", "scripts/collect_scripted.py",
            "scripts/prepare_vla_training.py", "scripts/train_vla.py", "scripts/finetune_planner.py",
            "scripts/evaluate_planner.py", "scripts/run_research_lab.py", "vlapipe/student_design.py"]
missing = [p for p in required if not (PROJECT / p).is_file()]
if missing:
    raise RuntimeError("선택한 코드에 필요한 최신 파일이 없습니다: " + str(missing) +
                       " — 최신 브랜치 push 또는 Upload to Colab 후 SOURCE_ROOT를 확인하세요.")
print("원격 실행 코드:", PROJECT)


## 4. 공통 실행기 — 로그·오류·백업을 한곳에

각 단계는 새 출력 폴더를 사용합니다. 실패도 로그와 종료 코드에 남기고 예외를 숨기지 않습니다. 단계 종료와 오류 시 Drive로 복사합니다. 실행 도중 갑자기 런타임 자체가 종료되면 마지막 단계의 미복사 파일은 잃을 수 있으므로 긴 작업은 저장 주기를 별도로 설정해야 합니다. 자동으로 세션을 유지하는 코드는 사용하지 않습니다.


In [ ]:
import hashlib, tarfile, csv
STAGES = {}
COMMANDS = []
BACKUP_SIGNATURES = {}

def run_files():
    # A Windows checkpoint 'last' junction must not duplicate GB-sized weights.
    for directory, dirs, names in os.walk(RUN):
        dirs[:] = sorted(d for d in dirs if d != "checkout" and not (Path(directory)/d).is_symlink()
                         and not (Path(directory)/d).is_junction())
        for name in sorted(names):
            p = Path(directory)/name
            if p.is_file() and not p.is_symlink():
                yield p

def dump(path, value):
    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")

def sync():
    for src in run_files():
        dst = PERSIST / src.relative_to(RUN)
        dst.parent.mkdir(parents=True, exist_ok=True)
        signature = (src.stat().st_size, src.stat().st_mtime_ns)
        if BACKUP_SIGNATURES.get(str(src)) != signature or not dst.exists() or dst.stat().st_size != signature[0]:
            tmp = dst.with_name(dst.name + ".copying")
            shutil.copy2(src, tmp)
            tmp.replace(dst)
            BACKUP_SIGNATURES[str(src)] = signature
    print("BACKUP:", PERSIST)

def stage(name):
    folder = RUN / (name + "_" + uuid.uuid4().hex[:6])
    STAGES[name] = folder
    dump(RUN / "stage_paths.json", {k: str(v.relative_to(RUN)) for k,v in STAGES.items()})
    return folder

def command(args, name, cwd=None):
    argv = [str(x) for x in args]
    log = RUN / "logs" / (name + "_" + uuid.uuid4().hex[:6] + ".log")
    log.parent.mkdir(exist_ok=True)
    record = {"name": name, "argv": argv, "cwd": str(cwd or PROJECT), "log": str(log.relative_to(RUN)), "status": "running"}
    COMMANDS.append(record)
    dump(RUN / "commands.json", COMMANDS)
    proc = None
    try:
        with log.open("w", encoding="utf-8") as stream:
            proc = subprocess.Popen(argv, cwd=cwd or PROJECT, env=dict(os.environ, PYTHONUNBUFFERED="1"),
                                    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                                    encoding="utf-8", errors="replace")
            for line in proc.stdout:
                stream.write(line); stream.flush(); print(line, end="", flush=True)
            rc = proc.wait()
        record.update(status="passed" if rc == 0 else "failed", returncode=rc)
        if rc:
            raise RuntimeError(f"{name} failed ({rc}). 로그 / log: {log}")
    except BaseException as exc:
        if proc is not None and proc.poll() is None:
            proc.terminate(); proc.wait(timeout=30)
        record.update(status="failed", error=type(exc).__name__)
        raise
    finally:
        dump(RUN / "commands.json", COMMANDS)
        sync()

def script(name, *args):
    command([PYTHON, PROJECT / "scripts" / name, *args], Path(name).stem)

os.environ.update(HF_HUB_DISABLE_XET="1", HF_HUB_DISABLE_PROGRESS_BARS="1", TOKENIZERS_PARALLELISM="false", PYTHONIOENCODING="utf-8")
if platform.system() == "Linux":
    os.environ.update(MUJOCO_GL="egl", PYOPENGL_PLATFORM="egl")
sys.path.insert(0, str(PROJECT))
from vlapipe.render_setup import configure_headless_rendering
dump(RUN / "renderer_setup.json", configure_headless_rendering(RUN))
git = subprocess.run(["git", "-C", str(PROJECT), "rev-parse", "HEAD"], capture_output=True, text=True)
status = subprocess.run(["git", "-C", str(PROJECT), "status", "--porcelain"], capture_output=True, text=True)
files = {}
with tarfile.open(RUN / "source_snapshot.tar.gz", "w:gz") as archive:
    for directory, dirs, names in os.walk(PROJECT):
        dirs[:] = sorted(d for d in dirs if d not in {".git", ".venv", "outputs", "data", "assets", "__pycache__"})
        for name in sorted(names):
            p = Path(directory) / name
            rel = p.relative_to(PROJECT)
            if p.suffix in {".py", ".txt", ".json", ".md", ".ipynb"}:
                files[str(rel)] = hashlib.sha256(p.read_bytes()).hexdigest()
                archive.add(p, arcname=str(rel))
dump(RUN / "source_manifest.json", {"commit": git.stdout.strip() if git.returncode == 0 else None,
     "dirty": bool(status.stdout.strip()) if status.returncode == 0 else None, "files": files})
dump(RUN / "configuration.json", {k: globals()[k] for k in ["TEAM", "PROFILE", "RUN_ID", "VLM_MODEL", "PIPELINE_CHECK_VLM", "VLA_BASE", "VLM_STEPS", "VLA_STEPS", "VLA_BATCH", "COUNT", "COLLECT_SEED", "COLLECT_SCENES", "HOLDOUT_SEEDS", "VLM_TRAIN_SEEDS", "VLM_VAL_SEEDS", "VLM_TEST_SEEDS", "EVALUATE_TEST"]})
sync()


## 5. 격리된 Python 환경 설치

Colab 기본 Python과 라이브러리는 바뀔 수 있습니다. 노트북 커널은 그대로 두고 **원격 Python 3.12 환경**에서 모든 프로젝트 스크립트를 실행합니다. 그래서 기본 torch를 덮어쓰거나 커널을 여러 번 재시작할 필요가 없습니다. 처음 설치와 모델 다운로드에는 시간이 걸립니다. 출력의 pip 오류를 건너뛰지 마세요.


In [ ]:
ENV_DIR = Path("/content/Final_project_env") if platform.system() == "Linux" else RUN / "env"
if PYTHON_OVERRIDE:
    PYTHON = Path(PYTHON_OVERRIDE)
elif INSTALL:
    command([sys.executable, "-m", "pip", "install", "uv==0.13.7"], "install_uv")
    command([sys.executable, "-m", "uv", "venv", "--python", "3.12", str(ENV_DIR)], "create_environment") if not ENV_DIR.exists() else None
    PYTHON = ENV_DIR / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
    command([sys.executable, "-m", "uv", "pip", "install", "--python", PYTHON,
             "--index-strategy", "unsafe-best-match", "--extra-index-url", "https://download.pytorch.org/whl/cu128",
             "-r", PROJECT / "requirements-colab-lock.txt"], "install_project")
else:
    PYTHON = Path(sys.executable)
command([sys.executable, "-m", "uv", "pip", "check", "--python", PYTHON], "pip_check") if INSTALL and not PYTHON_OVERRIDE else None
diagnostic = RUN / "diagnose_environment.py"
diagnostic.write_text("import sys,json,platform,torch,mujoco,importlib.metadata as m\nfrom pathlib import Path\n"
    "r={'python':sys.version,'platform':platform.platform(),'torch':torch.__version__,'cuda':torch.cuda.is_available(),"
    "'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,"
    "'vram_bytes':torch.cuda.get_device_properties(0).total_memory if torch.cuda.is_available() else None,"
    "'bf16':torch.cuda.is_bf16_supported(including_emulation=False) if torch.cuda.is_available() else False,"
    "'packages':{x:m.version(x) for x in ['mujoco','transformers','lerobot','numpy','h5py']}}\n"
    "if sys.platform.startswith('linux'):\n"
    " from OpenGL import GL\n c=mujoco.GLContext(64,64); c.make_current()\n"
    " r['gl_vendor']=GL.glGetString(GL.GL_VENDOR).decode(); r['gl_renderer']=GL.glGetString(GL.GL_RENDERER).decode(); c.free()\n"
    f"Path({str(RUN / 'environment.json')!r}).write_text(json.dumps(r,indent=2),encoding='utf-8')\nprint(json.dumps(r,indent=2))\n"
    + ("assert r['cuda'], 'GPU가 없습니다. Colab GPU 런타임을 선택하세요.'\n" if RUN_ML else ""), encoding="utf-8")
command([PYTHON, diagnostic], "environment")
ENVIRONMENT = json.loads((RUN / "environment.json").read_text())
command([PYTHON, "setup_assets.py"], "panda_assets")


## 6. 기본 제어와 저장 검사

학생 설계와 무관하게 통과해야 하는 부분입니다. 실패하면 연구 가설을 바꾸지 말고 오류 로그를 조교에게 전달합니다. 실제 영상·로그를 확인한 뒤 데이터 수집으로 갑니다. 기본 검사의 역방향 배열 검사는 역방향 시연의 물리적 성공을 보장하지 않습니다.


In [ ]:
script("check_planner_scoring.py")
SMOKE = stage("smoke")
script("smoke_test.py", "--out", SMOKE)
BASELINE = stage("baseline")
script("run_pipeline.py", "--objects", "number", "--number-count", COUNT,
       "--layout", "messy", "--seed", 3, "--planner", "rule", "--policy", "scripted", "--out", BASELINE)


## 7. 결과 영상 찾기

노트북 출력은 빠른 확인용입니다. 원본은 RUN 폴더와 Drive 백업에 있습니다. 영상이 생성되지 않았다면 보고서와 단계 로그부터 읽습니다. 실패 영상도 지우지 않습니다.


In [ ]:
from IPython.display import display, Image, Markdown, FileLink
report_paths = list(BASELINE.rglob("report.json"))
for p in report_paths:
    print(p, json.loads(p.read_text(encoding="utf-8")).get("summary"))
for p in BASELINE.rglob("rollout.gif"):
    display(Image(filename=str(p)))


## 8. 고전 제어기로 직접 정리 데이터 수집

이미 아는 물체 위치로 접근→파지→들기→운반→놓기를 실제 시뮬레이션에서 실행합니다. 성공 시연은 `accepted`, 실패는 `rejected`, 전체 분모는 `collection_report.json`에 남습니다. connection에서는 장면마다 운반 하나만 수집하므로 전체 정리 시연과 구분하세요.

사람의 조이스틱·키보드는 Windows PC에 연결되어 있고 Colab은 원격이므로 pygame 창이 로컬 입력을 직접 받지 못합니다. 이 노트북은 자동 수집을 기본으로 합니다. 사람 수집은 `TELEOP_STUDENT.md`의 로컬 CPU 수집기로 녹화·검토한 **accepted 파일만** Colab 확장으로 업로드한 뒤 아래 `EXTRA_ACCEPTED_DIR`에 원격 경로를 넣습니다. 장치 입력을 원격에 연결했다고 주장하지 않습니다.


In [ ]:
COLLECTION = stage("collection")
script("collect_scripted.py", "--objects", "number", "--count", COUNT, "--direction", "tidy",
       "--task-level", "subtask", "--seed", COLLECT_SEED, "--episodes", COLLECT_SCENES,
       "--max-transfers", 1 if PROFILE == "connection" else 0, "--out-dir", COLLECTION)
accepted_dirs = sorted({p.parent for p in COLLECTION.rglob("accepted/*.hdf5")})
if not accepted_dirs:
    raise RuntimeError("성공 시연이 없습니다. collection_report.json의 실패를 먼저 분석하세요.")
EXTRA_ACCEPTED_DIR = ""  # 선택: 사람이 검토한 원격 HDF5 폴더. rejected 전체를 넣지 않기
if EXTRA_ACCEPTED_DIR:
    accepted_dirs.append(Path(EXTRA_ACCEPTED_DIR))
print("학습 후보:", accepted_dirs)
PREVIEW = stage("demo_preview"); PREVIEW.mkdir()
first_demo = next(accepted_dirs[0].glob("*.hdf5"))
script("preview_episode.py", first_demo, "--out", PREVIEW / "agentview.gif")
script("preview_episode.py", first_demo, "--camera", "robot0_eye_in_hand", "--out", PREVIEW / "wrist.gif")
display(Image(filename=str(PREVIEW / "agentview.gif")))
display(Image(filename=str(PREVIEW / "wrist.gif")))


## 9. VLM용 이미지–정답 데이터

VLM은 이미지와 명령에서 물체·목적지 JSON을 학습합니다. VLA는 두 카메라·상태16에서 액션7을 학습합니다. 이미지 JSON만 모아서는 VLA 시연이 되지 않습니다. 여기서는 시뮬레이터에서 만든 알려진 구역 규칙을 정답으로 쓰며, 이를 모델이 스스로 발견한 접근성 정답이라고 해석하지 않습니다.

이 셀은 train/validation/test를 원본 seed로 분리합니다. 시연에서 VLM 데이터를 만들려면 `scripts/episodes_to_planner_dataset.py`를 사용하고 같은 원본 seed의 모든 변형을 같은 분할에 둡니다.


In [ ]:
PLANNER_DATA = stage("planner_data")
script("collect_planner_dataset.py", "--out", PLANNER_DATA, "--objects", "number", "--count", COUNT,
       "--layouts", "messy", "cluttered", "--prompt-style", "number", "--image-size", 512,
       "--train-seeds", VLM_TRAIN_SEEDS, "--val-seeds", VLM_VAL_SEEDS, "--test-seeds", VLM_TEST_SEEDS)
script("finetune_planner.py", "--dataset", PLANNER_DATA, "--out", stage("planner_dry_run"), "--dry-run")


## 10. VLM 파인튜닝과 저장 모델 재평가

같은 validation에서 base와 저장 모델을 비교합니다. 출력 형식 정확성, 물체·목적지 F1, 완전 일치와 원문 출력을 함께 봅니다. loss 감소를 실제 운반 성공으로 바꾸어 말하지 않습니다. `EVALUATE_TEST=True`는 설정과 모델 선택을 끝낸 뒤에만 사용합니다. GPU 부족이면 batch가 이미 1이므로 이미지 해상도·모델·학습 방식을 검토하고 조건 변경을 기록합니다.


In [ ]:
VLM_TRAIN = None
if RUN_ML:
    SPLIT = "test" if EVALUATE_TEST else "validation"
    BASE_EVAL = stage("vlm_base_eval")
    script("evaluate_planner.py", "--dataset", PLANNER_DATA, "--split", SPLIT,
           "--model", VLM_MODEL, "--device", "cuda", "--max-new-tokens", 512, "--out", BASE_EVAL)
    VLM_TRAIN = stage("vlm_train")
    precision = ["--autocast-bf16"] if ENVIRONMENT["bf16"] else []
    script("finetune_planner.py", "--dataset", PLANNER_DATA, "--model", VLM_MODEL,
           "--max-steps", VLM_STEPS, "--device", "cuda", "--gradient-checkpointing", "--out", VLM_TRAIN, *precision)
    SAVED_VLM = VLM_TRAIN / "pretrained_model"
    FT_EVAL = stage("vlm_saved_eval")
    script("evaluate_planner.py", "--dataset", PLANNER_DATA, "--split", SPLIT,
           "--model", SAVED_VLM, "--device", "cuda", "--max-new-tokens", 512, "--out", FT_EVAL)
else:
    print("명시적으로 ML을 생략했습니다. 학습·저장 모델 평가를 통과한 것이 아닙니다.")


## 11. VLA 시연 분할·변환·파인튜닝

성공 원본 시연을 seed로 나눈 뒤 train만 LeRobot으로 변환합니다. holdout은 학습에 넣지 않습니다. 이 간단한 실행기는 train/holdout 2분할이므로, 연구에서 학습량을 선택할 때는 별도 validation 장면을 확보하고 최종 test를 반복 사용하지 마세요. 2스텝 weight도 저장·재로딩할 수 있으나 작업 학습이 충분하다는 뜻은 아닙니다.


In [ ]:
VLA_TRAIN = None
if RUN_ML:
    VLA_DATA = stage("vla_data")
    input_flags = [x for d in accepted_dirs for x in ("--input-dir", str(d))]
    script("prepare_vla_training.py", *input_flags, "--system", "hier", "--holdout-seeds",
           ",".join(map(str, HOLDOUT_SEEDS)), "--out", VLA_DATA, "--repo-id", "local/final_project",
           "--convert", "--images")
    VLA_TRAIN = stage("vla_train")
    script("train_vla.py", "--dataset-root", VLA_DATA / "lerobot", "--repo-id", "local/final_project",
           "--base", VLA_BASE, "--steps", VLA_STEPS, "--batch-size", VLA_BATCH, "--out", VLA_TRAIN)
    checkpoints = sorted((VLA_TRAIN / "checkpoints").glob("[0-9]*/pretrained_model"))
    if not checkpoints:
        raise RuntimeError("저장된 VLA checkpoint가 없습니다. 학습 로그를 확인하세요.")
    SAVED_VLA = checkpoints[-1]
    print("VLA weight와 전후처리 저장 폴더:", SAVED_VLA)


## 12. 저장 VLM → 저장 VLA → 실제 시뮬레이터

여기서는 저장 모델을 다시 불러옵니다. connection은 짧은 예산으로 입출력 연결만 확인하므로 성공률 평가에 사용하지 않습니다. study에서는 독립 장면·같은 예산의 기준선과 비교할 설계를 먼저 만드세요. 빈 VLM 출력을 정답 계획으로 채우는 fallback은 사용하지 않습니다.

첫 시험은 알려진 계획으로 저장 VLA가 실제 액션을 내는지 확인합니다. 둘째는 별도 base Qwen VLM→저장 VLA 연결입니다. 이 VLM은 파인튜닝하지 않았고, 연구 실행기의 좌표·구역 규칙 문맥을 함께 보므로 순수 영상 인식 성능이 아닙니다. 셋째는 이번에 학습·저장한 VLM→저장 VLA입니다. 짧게 학습한 VLM이 형식을 지키지 못하면 0액션으로 끝날 수 있습니다. 이때 둘째 시험의 결과를 셋째 시험의 성공으로 바꾸어 쓰지 마세요.


In [ ]:
if RUN_ML:
    VLA_PROBE = stage("vla_saved_probe")
    script("run_research_lab.py", "--mode", "A", "--case", "open", "--seed", 0,
           "--planner", "oracle_transport", "--policy", "smolvla", "--checkpoint", SAVED_VLA,
           "--max-steps", 3 if PROFILE == "connection" else 400, "--max-attempts", 1,
           "--total-budget", 12 if PROFILE == "connection" else 1200, "--out", VLA_PROBE)
    BASE_VLM_CONNECTION = stage("base_vlm_vla_connection")
    script("run_research_lab.py", "--mode", "A", "--case", "open", "--seed", 0,
           "--planner", "vlm", "--vlm-model", PIPELINE_CHECK_VLM, "--prompt-style", "number",
           "--policy", "smolvla", "--checkpoint", SAVED_VLA,
           "--max-steps", 3 if PROFILE == "connection" else 400, "--max-attempts", 1,
           "--total-budget", 12 if PROFILE == "connection" else 1200, "--out", BASE_VLM_CONNECTION)
    LEARNED = stage("saved_pipeline")
    script("run_pipeline.py", "--objects", "number", "--number-count", COUNT,
           "--layout", "messy", "--seed", 6, "--planner", "vlm", "--vlm-model", SAVED_VLM,
           "--prompt-style", "number", "--vlm-image-size", 512,
           "--policy", "smolvla", "--checkpoint", SAVED_VLA, "--passes", 1,
           "--max-steps", 3 if PROFILE == "connection" else 500, "--out", LEARNED)
    for p in LEARNED.rglob("report.json"):
        print(json.loads(p.read_text(encoding="utf-8")).get("summary"))
    for p in LEARNED.rglob("rollout.gif"):
        display(Image(filename=str(p)))


## 13. 학생 연구 설계 연결

기본 A는 동작합니다. B/C/D는 학생이 `vlapipe/student_design.py`의 세 함수를 구현한 뒤 사용합니다.

|조건|실행 전 검사·선택|실패 후 재계획|
|---|---|---|
|A|없음|없음|
|B|학생 설계|없음|
|C|없음|학생 설계|
|D|학생 설계|학생 설계|

**먼저 예측을 적으세요.** ①어떤 조건에서 차이가 날까 ②비교 상대는 무엇인가 ③반대 결과라면 무엇을 다시 생각할까. 검사와 재계획의 효과를 구분하려면 같은 초기 배치·관측 정보·총 행동 예산·물체별 시도 상한을 사용합니다. 단순히 시도 횟수를 더 준 효과인지도 확인합니다.

다음 셀은 원격에서 실제 실행할 학생 코드를 표시합니다. 로컬에서 편집했다면 push한 브랜치에서 새로 받거나 Upload to Colab 후 그 파일을 지정합니다. 저장한 코드와 실행한 코드를 혼동하지 마세요.


In [ ]:
STUDENT_DESIGN = PROJECT / "vlapipe/student_design.py"
print(STUDENT_DESIGN.read_text(encoding="utf-8"))
# 선택: Upload to Colab으로 올린 완성 파일의 원격 경로로 변경
# STUDENT_DESIGN = Path("/content/my_student_design.py")
DESIGN_COPY = RUN / "student_design_used.py"
shutil.copy2(STUDENT_DESIGN, DESIGN_COPY)
print("실험에 사용할 고정 사본:", DESIGN_COPY)
sync()


## 14. 공정한 비교 실행

처음에는 A만 실행합니다. 세 함수를 완성하면 `MODES`를 늘립니다. `open/blocked/destination_blocked/unreachable`은 배관 검사에 쓰는 소수의 고정 배치이며 seed가 달라도 같은 배치가 반복될 수 있습니다. 이를 수십 개의 독립 장면으로 세지 마세요. 연구 평가는 messy/cluttered 등의 배치 다양성과 초기 geometry hash를 확인하고, 필요한 조건 생성기를 별도로 설계합니다. 도달 불가 거절은 작업 완료와 다릅니다.


In [ ]:
MODES = ["A"]          # 완성 후 ["A", "B", "C", "D"]
CASES = ["open", "destination_blocked"]
SEEDS = [0]            # 진단용. 연구의 독립 반복 수가 아님
TRIAL_ROOT = stage("research"); TRIAL_ROOT.mkdir()
for case in CASES:
    for seed in SEEDS:
        for mode in MODES:
            extra = ["--design-file", DESIGN_COPY] if mode != "A" else []
            script("run_research_lab.py", "--mode", mode, "--case", case, "--seed", seed,
                   "--planner", "oracle_transport", "--policy", "scripted", "--count", 4,
                   "--total-budget", 1200, "--max-steps", 400, "--max-attempts", 2,
                   "--out", TRIAL_ROOT / f"{case}_{seed}_{mode}", *extra)


## 15. 결과 취합과 해석

이 표는 실행 결과를 정리하며 연구 가설을 자동 판정하지 않습니다. 같은 geometry hash를 가진 반복은 독립 배치로 세지 않습니다. 실패 장면을 빼고 시간 평균을 내면 유리하게 보일 수 있으므로 전체 완료율과 실패·예산 소진을 함께 보고합니다. 모델 호출 시간과 동작 단계 수도 서로 다른 비용입니다.


In [ ]:
rows = []
for p in TRIAL_ROOT.rglob("report.json"):
    r = json.loads(p.read_text(encoding="utf-8"))
    row = {k:r.get(k) for k in ["mode", "case", "seed", "initial_geometry_sha256", "stop_reason"]}
    row.update(r.get("summary", {})); row["report"] = str(p.relative_to(RUN))
    for key in ["attempts", "final_misplaced"]:
        if key in row: row[key] = json.dumps(row[key], ensure_ascii=False)
    rows.append(row)
if rows:
    with (RUN / "results.csv").open("w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=sorted(set().union(*(r.keys() for r in rows))))
        writer.writeheader(); writer.writerows(rows)
for row in rows: print(json.dumps(row, ensure_ascii=False))
hashes = [r["initial_geometry_sha256"] for r in rows if r.get("initial_geometry_sha256")]
print("서로 다른 초기 배치 hash:", len(set(hashes)), "(조건 사이 같은 배치 공유는 의도한 비교)")
sync()


## 16. 산출물 색인과 모델 배포 명세

`ARTIFACTS.md`에서 영상·로그·데이터·weight 위치를 찾습니다. 각 파일의 SHA256도 저장합니다. 모델은 weight 파일 하나만 보내지 말고 `pretrained_model` 전체, 기반 모델 ID/revision, 코드·학습 설정·분할·평가 결과를 함께 전달합니다. Git에는 코드와 작은 명세를, 큰 모델은 공유 Drive 등에 보관합니다.

중단 뒤에는 Drive의 RUN_ID 폴더부터 확인합니다. 이 노트북은 학습 optimizer의 자동 재개 기능을 제공하지 않으므로 이전 weight를 발견했다고 이어 학습했다고 표시하지 않습니다. 새 실행은 새 RUN_ID로 시작합니다.


In [ ]:
artifact_rows = []
for p in run_files():
    if p.name in {"artifact_manifest.json", "ARTIFACTS.md"}: continue
    sha = hashlib.sha256()
    with p.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024*1024), b""): sha.update(chunk)
    artifact_rows.append({"path": str(p.relative_to(RUN)), "bytes": p.stat().st_size, "sha256": sha.hexdigest()})
dump(RUN / "artifact_manifest.json", {"run_id": RUN_ID, "profile": PROFILE,
     "claim": "connection check only" if PROFILE == "connection" else "evaluate research claims separately",
     "files": artifact_rows})
lines = ["# Final_project 산출물", "", f"RUN_ID: {RUN_ID}", f"작업 폴더: {RUN}", f"백업 폴더: {PERSIST}", "",
         "## 찾는 방법", "stage_paths.json: 단계별 폴더", "commands.json / logs/: 실행 명령·오류", 
         "source_manifest.json / source_snapshot.tar.gz: 실행 코드", "results.csv: 연구 실행 취합", "",
         "## 파일 목록", ""]
lines += [f"- `{r['path']}` ({r['bytes']:,} bytes)" for r in artifact_rows]
(RUN / "ARTIFACTS.md").write_text("\n".join(lines), encoding="utf-8")
sync()
display(Markdown((RUN / "ARTIFACTS.md").read_text(encoding="utf-8")))
print("완료. 보관 폴더에서 이 RUN_ID와 ARTIFACTS.md를 확인하세요:", PERSIST)


## 막혔을 때 조교에게 전달할 것

RUN_ID, 실행한 셀, `environment.json`, `commands.json`, 실패 단계의 `.log`, 관련 보고서와 영상 경로를 전달합니다. 비밀번호·토큰은 보내지 않습니다.

- GPU가 없음: 로컬 Windows/Colab CPU 커널을 선택했는지 확인.
- 최신 파일이 없음: 로컬 수정만 있고 원격 Git 브랜치에는 없는지 확인.
- EGL 오류: Linux GPU 드라이버·렌더링 설치 문제. 기본 검사 이전에는 학습하지 않기.
- CUDA OOM: 다른 모델 프로세스가 남았는지 확인. 메모리·해상도·학습 방식 변경을 기록.
- 성공 시연이 없음: rejected와 수집 보고서에서 위치·파지·놓기 실패를 분리.
- VLM 계획이 비었음: 원문과 형식 오류를 확인. 정답 계획으로 몰래 채우지 않기.
- 학생 함수 NotImplemented: 플랫폼 고장이 아니라 아직 구현하지 않은 연구 판단.
- 차이가 없음: 곧바로 동등하다고 쓰지 말고 표본·조건·기전과 불확실성을 점검.

공식 환경 안내: [Colab VS Code 사용법](https://github.com/googlecolab/colab-vscode/wiki/User-Guide), [Colab FAQ](https://research.google.com/colaboratory/faq.html). GPU 종류·사용 가능 시간은 할당에 따라 달라집니다. 이 노트북의 자동 검증 기록은 조교 문서 `FINAL_PROJECT_VALIDATION.md`에서 실행 환경과 미검증 범위를 구분합니다.
